# Tratamento e Enriquecimento dos Dados

## Objetivo

Transformar os dados brutos coletados via API em uma base analítica estruturada para utilização em modelos de inteligência de mercado.

## Etapas Executadas

* Tratamento de tipos de dados
* Padronização de colunas
* Conversão de valores numéricos
* Cruzamento de bases
* Criação do indicador de PIB per capita
* Inclusão de variáveis territoriais

## Aplicação de Negócio

A qualidade dos resultados depende diretamente da qualidade dos dados utilizados. Esta etapa garante consistência e confiabilidade para análises posteriores.

## Resultado Esperado

Base analítica consolidada contendo:

* PIB Municipal
* PIB per capita
* População Residente
* Área Territorial
* Densidade Demográfica


In [ ]:
import pandas as pd

caminho_arquivo = "dados/brutos/renda_municipios_ibge.csv"

df_dados_brutos = pd.read_csv(
    caminho_arquivo,
    encoding="utf-8-sig"
)

print("Dimensões da base:")
print(df_dados_brutos.shape)

display(df_dados_brutos.head())

Dimensões da base:
(5571, 11)


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
1,6,Município,40,Mil Reais,919520,1100015,Alta Floresta D'Oeste - RO,37,Produto Interno Bruto a preços correntes,2022,2022
2,6,Município,40,Mil Reais,3809355,1100023,Ariquemes - RO,37,Produto Interno Bruto a preços correntes,2022,2022
3,6,Município,40,Mil Reais,289783,1100031,Cabixi - RO,37,Produto Interno Bruto a preços correntes,2022,2022
4,6,Município,40,Mil Reais,3195489,1100049,Cacoal - RO,37,Produto Interno Bruto a preços correntes,2022,2022


In [ ]:
df_pib_municipios = df_dados_brutos.copy()

df_pib_municipios = df_pib_municipios.iloc[1:].copy()

novos_nomes_colunas = {
    "V": "pib_municipal",
    "D1C": "codigo_municipio",
    "D1N": "municipio",
    "D3N": "ano"
}

df_pib_municipios = df_pib_municipios.rename(
    columns=novos_nomes_colunas
)

df_pib_municipios["pib_municipal"] = pd.to_numeric(
    df_pib_municipios["pib_municipal"],
    errors="coerce"
)

print("Dimensões da base limpa:")
print(df_pib_municipios.shape)

print("\nTipos de dados:")
print(
    df_pib_municipios[
        ["codigo_municipio", "municipio", "pib_municipal", "ano"]
    ].dtypes
)

display(
    df_pib_municipios[
        ["codigo_municipio", "municipio", "pib_municipal", "ano"]
    ].head()
)

Dimensões da base limpa:
(5570, 11)

Tipos de dados:
codigo_municipio    object
municipio           object
pib_municipal        int64
ano                 object
dtype: object


,codigo_municipio,municipio,pib_municipal,ano
1,1100015,Alta Floresta D'Oeste - RO,919520,2022
2,1100023,Ariquemes - RO,3809355,2022
3,1100031,Cabixi - RO,289783,2022
4,1100049,Cacoal - RO,3195489,2022
5,1100056,Cerejeiras - RO,903099,2022


In [ ]:
import requests
import pandas as pd

url_api_populacao = (
    "https://apisidra.ibge.gov.br/values/"
    "t/6579/n6/all/v/9324/p/2022?formato=json"
)

resposta = requests.get(url_api_populacao)

print("Status:", resposta.status_code)

if resposta.status_code == 200:

    dados_populacao = resposta.json()

    df_populacao_municipios = pd.DataFrame(dados_populacao)

    print("\nDimensões:")
    print(df_populacao_municipios.shape)

    display(df_populacao_municipios.head())

else:

    print(resposta.text)

Status: 200

Dimensões:
(1, 11)


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano


In [ ]:
# — Diagnóstico da API de População ──────────────────────────────
# COMO funciona: exibe o conteúdo bruto retornado pelo SIDRA.
# ONDE é usado: validação da consulta antes da transformação dos dados.
# POR QUE esta escolha: precisamos confirmar se a tabela possui dados.
# QUANDO rodar: quando a API retorna apenas cabeçalhos.
# O QUE produz: resposta textual completa da API.

import requests

url_api_populacao = (
    "https://apisidra.ibge.gov.br/values/"
    "t/6579/n6/all/v/9324/p/2022?formato=json"
)

resposta = requests.get(url_api_populacao)

print("Status:", resposta.status_code)

print("\nPrimeiros 1000 caracteres da resposta:")
print(resposta.text[:1000])

Status: 200

Primeiros 1000 caracteres da resposta:
[{"NC":"Nível Territorial (Código)","NN":"Nível Territorial","MC":"Unidade de Medida (Código)","MN":"Unidade de Medida","V":"Valor","D1C":"Município (Código)","D1N":"Município","D2C":"Variável (Código)","D2N":"Variável","D3C":"Ano (Código)","D3N":"Ano"}]


In [ ]:
# — Teste da Base de População do Censo 2022 ──────────────────────────────
# COMO funciona: consulta uma tabela do Censo 2022 com dados populacionais.
# ONDE é usado: preparação para cálculo do PIB per capita.
# POR QUE esta escolha: a tabela de estimativas não possui dados para 2022.
# QUANDO rodar: após identificar falha na tabela 6579.
# O QUE produz: amostra da base populacional do Censo 2022.

import requests
import pandas as pd

url_api_populacao = (
    "https://apisidra.ibge.gov.br/values/"
    "t/4714/n6/all/v/allxp/p/2022?formato=json"
)

resposta = requests.get(url_api_populacao)

print("Status:", resposta.status_code)

if resposta.status_code == 200:

    dados_populacao = resposta.json()

    df_populacao = pd.DataFrame(dados_populacao)

    print("\nDimensões:")
    print(df_populacao.shape)

    display(df_populacao.head())

else:

    print(resposta.text)

Status: 200

Dimensões:
(16711, 11)


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
1,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022
2,6,Município,26,Quilômetros quadrados,7067.127,1100015,Alta Floresta D'Oeste - RO,6318,Área da unidade territorial,2022,2022
3,6,Município,28,Habitante por quilômetro quadrado,3.04,1100015,Alta Floresta D'Oeste - RO,614,Densidade demográfica,2022,2022
4,6,Município,45,Pessoas,96833,1100023,Ariquemes - RO,93,População residente,2022,2022


In [ ]:
# — Identificação das Variáveis do Censo ──────────────────────────────
# COMO funciona: lista todas as variáveis retornadas pela tabela do Censo.
# ONDE é usado: seleção da variável correta para cálculo do PIB per capita.
# POR QUE esta escolha: a tabela possui múltiplos indicadores por município.
# QUANDO rodar: após validar a coleta da tabela 4714.
# O QUE produz: lista de variáveis disponíveis na base.

print("Variáveis encontradas:\n")

for variavel in sorted(df_populacao["D2N"].unique()):
    print(variavel)

Variáveis encontradas:

Densidade demográfica
População residente
Variável
Área da unidade territorial


In [ ]:
# — Filtragem da População Residente ──────────────────────────────
# COMO funciona: mantém apenas os registros da variável População residente.
# ONDE é usado: preparação para o cruzamento com a base de PIB.
# POR QUE esta escolha: as demais variáveis (área e densidade) não participam
# do cálculo do PIB per capita.
# QUANDO rodar: após identificar as variáveis disponíveis na tabela do Censo.
# O QUE produz: DataFrame contendo apenas a população dos municípios.

df_populacao_residente = (
    df_populacao[
        df_populacao["D2N"] == "População residente"
    ]
    .copy()
)

print("Dimensões da base de população:")
print(df_populacao_residente.shape)

print("\nQuantidade de municípios únicos:")
print(df_populacao_residente["D1C"].nunique())

display(
    df_populacao_residente[
        ["D1C", "D1N", "V"]
    ].head()
)

Dimensões da base de população:
(5570, 11)

Quantidade de municípios únicos:
5570


,D1C,D1N,V
1,1100015,Alta Floresta D'Oeste - RO,21494
4,1100023,Ariquemes - RO,96833
7,1100031,Cabixi - RO,5351
10,1100049,Cacoal - RO,86887
13,1100056,Cerejeiras - RO,15890


In [ ]:
df_populacao_residente = df_populacao_residente.rename(
    columns={
        "D1C": "codigo_municipio",
        "D1N": "municipio_populacao",
        "V": "populacao_residente"
    }
)

df_populacao_residente["codigo_municipio"] = (
    df_populacao_residente["codigo_municipio"]
    .astype(str)
)

df_pib_municipios["codigo_municipio"] = (
    df_pib_municipios["codigo_municipio"]
    .astype(str)
)

df_populacao_residente["populacao_residente"] = pd.to_numeric(
    df_populacao_residente["populacao_residente"],
    errors="coerce"
)

df_municipios = df_pib_municipios.merge(
    df_populacao_residente[
        [
            "codigo_municipio",
            "populacao_residente"
        ]
    ],
    on="codigo_municipio",
    how="left"
)

print("Dimensões após o cruzamento:")
print(df_municipios.shape)

print("\nMunicípios sem população:")
print(
    df_municipios["populacao_residente"]
    .isna()
    .sum()
)

display(
    df_municipios[
        [
            "codigo_municipio",
            "municipio",
            "pib_municipal",
            "populacao_residente"
        ]
    ].head()
)

Dimensões após o cruzamento:
(5570, 12)

Municípios sem população:
0


,codigo_municipio,municipio,pib_municipal,populacao_residente
0,1100015,Alta Floresta D'Oeste - RO,919520,21494
1,1100023,Ariquemes - RO,3809355,96833
2,1100031,Cabixi - RO,289783,5351
3,1100049,Cacoal - RO,3195489,86887
4,1100056,Cerejeiras - RO,903099,15890


# Insight Metodológico 01 — Integração de Bases Públicas

## Evidência

O cruzamento entre a base de Produto Interno Bruto Municipal e a base de População Residente do Censo 2022 foi realizado para os 5.570 municípios brasileiros sem perda de registros.

## Relevância

A ausência de municípios sem correspondência indica consistência entre as bases utilizadas e reduz o risco de distorções nas análises posteriores.

## Aplicação no Projeto

A partir desta etapa torna-se possível criar indicadores derivados que não podem ser observados diretamente em uma única fonte de dados, como:

- PIB per capita;
- capacidade econômica relativa;
- potencial de consumo por habitante;
- score econômico municipal.

## Próxima Validação

Verificar se municípios que aparecem entre os maiores PIBs nacionais também mantêm posição de destaque quando a atividade econômica é ajustada pelo tamanho da população.

In [ ]:
#  — Cálculo do PIB Per Capita ──────────────────────────────
# COMO funciona: divide o PIB municipal pela população residente.
# ONDE é usado: identificação da riqueza relativa dos municípios.
# POR QUE esta escolha: o PIB absoluto favorece municípios populosos.
# QUANDO rodar: após o cruzamento entre PIB e população.
# O QUE produz: coluna pib_per_capita para análise econômica comparativa.

df_municipios["pib_per_capita"] = (
    df_municipios["pib_municipal"]
    / df_municipios["populacao_residente"]
)

print("Estatísticas do PIB per capita:")

print(
    df_municipios["pib_per_capita"]
    .describe()
)

display(
    df_municipios[
        [
            "municipio",
            "pib_municipal",
            "populacao_residente",
            "pib_per_capita"
        ]
    ].head()
)

Estatísticas do PIB per capita:
count    5570.000000
mean       36.717446
std        41.202673
min         6.757458
25%        14.914225
50%        26.263668
75%        44.302859
max       842.004600
Name: pib_per_capita, dtype: float64


,municipio,pib_municipal,populacao_residente,pib_per_capita
0,Alta Floresta D'Oeste - RO,919520,21494,42.780311
1,Ariquemes - RO,3809355,96833,39.339430
2,Cabixi - RO,289783,5351,54.154924
3,Cacoal - RO,3195489,86887,36.777527
4,Cerejeiras - RO,903099,15890,56.834424


# Insight de Negócio 03 — Riqueza por Habitante é Muito Mais Concentrada que o PIB Absoluto

## Evidência

O PIB per capita mediano dos municípios brasileiros é de aproximadamente R$ 26 mil por habitante.

Entretanto, alguns municípios atingem valores superiores a R$ 842 mil por habitante, mais de 30 vezes acima da mediana nacional.

## Descoberta

O tamanho da economia municipal não é suficiente para identificar mercados de maior capacidade econômica.

Municípios menores podem apresentar geração de riqueza por habitante significativamente superior à observada em grandes centros urbanos.

## Implicação para Inteligência de Mercado

Segmentações baseadas apenas em população ou PIB total podem deixar de identificar municípios com elevado potencial de consumo, crédito e investimento.

## Hipótese para Validação

Os municípios líderes em PIB per capita serão diferentes dos municípios líderes em PIB absoluto, revelando mercados de alta atratividade econômica que normalmente não aparecem em análises convencionais.

In [ ]:
# — Ranking dos Municípios por PIB Per Capita ──────────────────────────────
# COMO funciona: ordena os municípios pelo PIB per capita em ordem decrescente.
# ONDE é usado: identificação de mercados com alta capacidade econômica relativa.
# POR QUE esta escolha: o PIB per capita elimina o efeito do tamanho populacional.
# QUANDO rodar: após o cálculo da coluna pib_per_capita.
# O QUE produz: ranking dos 20 municípios com maior PIB per capita do Brasil.

df_top_20_pib_per_capita = (
    df_municipios[
        [
            "codigo_municipio",
            "municipio",
            "pib_municipal",
            "populacao_residente",
            "pib_per_capita"
        ]
    ]
    .sort_values(
        by="pib_per_capita",
        ascending=False
    )
    .head(20)
)

display(df_top_20_pib_per_capita)

print(
    "\nPIB per capita médio dos 20 municípios líderes:"
)

print(
    round(
        df_top_20_pib_per_capita["pib_per_capita"].mean(),
        2
    )
)

,codigo_municipio,municipio,pib_municipal,populacao_residente,pib_per_capita
3255,3305505,Saquarema - RJ,75409090,89559,842.004600
3215,3302700,Maricá - RJ,158394291,197277,802.902979
3499,3520400,Ilhabela - SP,24913501,34934,713.159129
2183,2929206,São Francisco do Conde - BA,25361789,38733,654.785041
3677,3536505,Paulínia - SP,70496584,110537,637.764586
3155,3204302,Presidente Kennedy - ES,8179738,13696,597.235543
5299,5107768,Santa Rita do Trivelato - MT,1409325,3276,430.196886
2410,3115359,Catas Altas - MG,2139893,5473,390.990864
3236,3304151,Quissamã - RJ,8493476,22393,379.291564
651,2112001,Tasso Fragoso - MA,3322534,8862,374.919206



PIB per capita médio dos 20 municípios líderes:
453.4


# Insight de Negócio 04 — Os Mercados Mais Atrativos Não São Necessariamente os Maiores

## Evidência

Os municípios com maior PIB per capita do Brasil são liderados por:

1. Saquarema (RJ) — R$ 842 mil por habitante
2. Maricá (RJ) — R$ 803 mil por habitante
3. Ilhabela (SP) — R$ 713 mil por habitante
4. São Francisco do Conde (BA) — R$ 655 mil por habitante
5. Paulínia (SP) — R$ 638 mil por habitante

O PIB per capita médio dos 20 municípios líderes foi de aproximadamente R$ 453 mil por habitante.

## Descoberta

Nenhum dos municípios do Top 5 faz parte do grupo das maiores populações do Brasil.

Isso demonstra que o tamanho do mercado e a riqueza gerada por habitante são variáveis distintas.

## Implicação para Inteligência de Mercado

Empresas que direcionam investimentos apenas para grandes centros urbanos podem ignorar mercados menores com elevada capacidade econômica.

Municípios com alta geração de riqueza por habitante tendem a apresentar maior potencial para:

- crédito premium;
- investimentos;
- seguros de maior valor;
- energia solar;
- consórcios;
- imóveis de médio e alto padrão.

## Evidência Contrária ao Senso Comum

São Paulo, Rio de Janeiro e Belo Horizonte dominam rankings de PIB absoluto, mas não aparecem entre os líderes de PIB per capita.

Isso sugere que volume econômico e eficiência econômica representam fenômenos diferentes.

## Hipótese para Próxima Validação

Municípios com elevado PIB per capita apresentarão maior volume de crédito concedido por habitante quando cruzados com dados do BACEN.

In [ ]:
# — Segmentação por Quartis de PIB Per Capita ──────────────────────────────
# COMO funciona: divide os municípios em 4 grupos com quantidades semelhantes.
# ONDE é usado: criação de segmentos econômicos para análise e priorização.
# POR QUE esta escolha: quartis reduzem a influência de valores extremos e permitem
# comparar municípios de forma padronizada.
# QUANDO rodar: após calcular e validar o PIB per capita.
# O QUE produz: coluna faixa_pib_per_capita com a classificação econômica.

df_municipios["faixa_pib_per_capita"] = pd.qcut(
    df_municipios["pib_per_capita"],
    q=4,
    labels=[
        "Baixo Potencial",
        "Potencial Médio",
        "Potencial Alto",
        "Potencial Muito Alto"
    ]
)

resumo_faixas = (
    df_municipios["faixa_pib_per_capita"]
    .value_counts()
    .sort_index()
)

print("Quantidade de municípios por faixa:\n")
print(resumo_faixas)

display(
    df_municipios[
        [
            "municipio",
            "pib_per_capita",
            "faixa_pib_per_capita"
        ]
    ].head(10)
)

Quantidade de municípios por faixa:

faixa_pib_per_capita
Baixo Potencial         1393
Potencial Médio         1392
Potencial Alto          1392
Potencial Muito Alto    1393
Name: count, dtype: int64


,municipio,pib_per_capita,faixa_pib_per_capita
0,Alta Floresta D'Oeste - RO,42.780311,Potencial Alto
1,Ariquemes - RO,39.339430,Potencial Alto
2,Cabixi - RO,54.154924,Potencial Muito Alto
3,Cacoal - RO,36.777527,Potencial Alto
4,Cerejeiras - RO,56.834424,Potencial Muito Alto
5,Colorado do Oeste - RO,32.830173,Potencial Alto
6,Corumbiara - RO,63.019683,Potencial Muito Alto
7,Costa Marques - RO,27.702225,Potencial Alto
8,Espigão D'Oeste - RO,31.324607,Potencial Alto
9,Guajará-Mirim - RO,26.858684,Potencial Alto


In [ ]:
# — Resumo Econômico por Faixa ──────────────────────────────
# COMO funciona: calcula estatísticas de PIB per capita para cada segmento.
# ONDE é usado: interpretação econômica das faixas criadas pelo qcut.
# POR QUE esta escolha: contar municípios não revela diferenças econômicas.
# QUANDO rodar: após criar a coluna faixa_pib_per_capita.
# O QUE produz: tabela-resumo para geração de insights executivos.

resumo_faixas = (
    df_municipios
    .groupby("faixa_pib_per_capita", observed=False)
    .agg(
        pib_per_capita_minimo=("pib_per_capita", "min"),
        pib_per_capita_medio=("pib_per_capita", "mean"),
        pib_per_capita_maximo=("pib_per_capita", "max")
    )
    .round(2)
)

display(resumo_faixas)

,pib_per_capita_minimo,pib_per_capita_medio,pib_per_capita_maximo
faixa_pib_per_capita,,,
Baixo Potencial,6.76,11.87,14.91
Potencial Médio,14.92,20.10,26.26
Potencial Alto,26.27,34.39,44.30
Potencial Muito Alto,44.30,80.50,842.00


# Insight de Negócio 05 — A Capacidade Econômica dos Municípios Não Cresce de Forma Linear

## Evidência

| Faixa | PIB per capita médio |
|---------|---------:|
| Baixo Potencial | R$ 11,9 mil |
| Potencial Médio | R$ 20,1 mil |
| Potencial Alto | R$ 34,4 mil |
| Potencial Muito Alto | R$ 80,5 mil |

## Descoberta

Os municípios classificados como Potencial Muito Alto possuem, em média, um PIB per capita 6,8 vezes superior aos municípios classificados como Baixo Potencial.

A diferença observada entre os extremos não representa apenas uma variação econômica normal, mas a existência de mercados estruturalmente distintos dentro do território nacional.

## Implicação para Inteligência de Mercado

Distribuir investimentos comerciais de forma homogênea entre municípios pode gerar desperdício de recursos.

Empresas que atuam com:

- crédito;
- seguros;
- energia solar;
- imóveis;
- consórcios;
- veículos;

podem aumentar eficiência comercial ao priorizar municípios posicionados na faixa de Potencial Muito Alto.

## Evidência Adicional

O grupo de Potencial Muito Alto apresenta elevada dispersão interna, chegando a municípios com PIB per capita superior a R$ 842 mil por habitante.

Isso indica a presença de polos econômicos altamente especializados que merecem análise individual.

In [ ]:
# — Salvamento da Base Processada ──────────────────────────────
# COMO funciona: cria a pasta de saída e salva a base tratada em CSV.
# ONDE é usado: encerramento da etapa de limpeza e transformação.
# POR QUE esta escolha: evita recalcular todo o pipeline a cada abertura do projeto.
# QUANDO rodar: após validar os indicadores e insights da base.
# O QUE produz: arquivo base_municipios_pib_per_capita.csv.

import os

pasta_processados = "dados/processados"

os.makedirs(
    pasta_processados,
    exist_ok=True
)

caminho_saida = (
    f"{pasta_processados}/"
    "base_municipios_pib_per_capita.csv"
)

df_municipios.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo com sucesso:")
print(caminho_saida)

print("\nQuantidade de registros:")
print(len(df_municipios))

Arquivo salvo com sucesso:
dados/processados/base_municipios_pib_per_capita.csv

Quantidade de registros:
5570


In [ ]:
# ─── Célula — Verificação das Variáveis Disponíveis no Censo ──────────────────────────────
# COMO funciona: lista todas as variáveis retornadas pela consulta do Censo.
# ONDE é usado: identificação de atributos adicionais para clusterização.
# POR QUE esta escolha: precisamos saber quais dimensões já estão disponíveis antes de buscar novas fontes.
# QUANDO rodar: após o carregamento da base do Censo.
# O QUE produz: lista das variáveis existentes na consulta.

print("Variáveis encontradas:")

for variavel in sorted(df_populacao["D2N"].unique()):
    print(variavel)

Variáveis encontradas:
Densidade demográfica
População residente
Variável
Área da unidade territorial


In [ ]:
df_populacao.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16711 entries, 0 to 16710
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   NC      16711 non-null  object
 1   NN      16711 non-null  object
 2   MC      16711 non-null  object
 3   MN      16711 non-null  object
 4   V       16711 non-null  object
 5   D1C     16711 non-null  object
 6   D1N     16711 non-null  object
 7   D2C     16711 non-null  object
 8   D2N     16711 non-null  object
 9   D3C     16711 non-null  object
 10  D3N     16711 non-null  object
dtypes: object(11)
memory usage: 1.4+ MB


In [ ]:
df_populacao["D2N"].value_counts()

D2N
População residente            5570
Área da unidade territorial    5570
Densidade demográfica          5570
Variável                          1
Name: count, dtype: int64

In [ ]:
# ─────────────────────────────────────────────────────────────
# ENRIQUECIMENTO DA BASE COM DADOS DO CENSO
# População + Área + Densidade
# ─────────────────────────────────────────────────────────────

import pandas as pd

# Converter valores para numérico
df_populacao["V"] = pd.to_numeric(
    df_populacao["V"],
    errors="coerce"
)

# População
df_pop = (
    df_populacao[
        df_populacao["D2N"] == "População residente"
    ][["D1C", "V"]]
    .rename(
        columns={
            "D1C": "codigo_municipio",
            "V": "populacao_residente"
        }
    )
)

# Área territorial
df_area = (
    df_populacao[
        df_populacao["D2N"] == "Área da unidade territorial"
    ][["D1C", "V"]]
    .rename(
        columns={
            "D1C": "codigo_municipio",
            "V": "area_km2"
        }
    )
)

# Densidade demográfica
df_densidade = (
    df_populacao[
        df_populacao["D2N"] == "Densidade demográfica"
    ][["D1C", "V"]]
    .rename(
        columns={
            "D1C": "codigo_municipio",
            "V": "densidade_demografica"
        }
    )
)

# Garantir mesmo tipo para merge
df_pop["codigo_municipio"] = df_pop["codigo_municipio"].astype(str)
df_area["codigo_municipio"] = df_area["codigo_municipio"].astype(str)
df_densidade["codigo_municipio"] = df_densidade["codigo_municipio"].astype(str)

df_municipios["codigo_municipio"] = (
    df_municipios["codigo_municipio"]
    .astype(str)
)

# Cruzamentos
df_municipios = (
    df_municipios
    .merge(
        df_area,
        on="codigo_municipio",
        how="left"
    )
    .merge(
        df_densidade,
        on="codigo_municipio",
        how="left"
    )
)

# Salvar nova base enriquecida
caminho_saida = (
    "dados/processados/"
    "base_municipios_enriquecida.csv"
)

df_municipios.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Base enriquecida criada com sucesso.")
print()
print("Dimensões:")
print(df_municipios.shape)

display(
    df_municipios[
        [
            "municipio",
            "pib_municipal",
            "populacao_residente",
            "area_km2",
            "densidade_demografica",
            "pib_per_capita"
        ]
    ].head()
)

Base enriquecida criada com sucesso.

Dimensões:
(5570, 16)


,municipio,pib_municipal,populacao_residente,area_km2,densidade_demografica,pib_per_capita
0,Alta Floresta D'Oeste - RO,919520,21494,7067.127,3.04,42.780311
1,Ariquemes - RO,3809355,96833,4426.571,21.88,39.339430
2,Cabixi - RO,289783,5351,1314.352,4.07,54.154924
3,Cacoal - RO,3195489,86887,3793.000,22.91,36.777527
4,Cerejeiras - RO,903099,15890,2783.300,5.71,56.834424
